# Topic 3: Linked Lists

**Goal**: Understand linked lists, build them from scratch, master the core operations.  
**Time**: ~4-5 hours  
**Prereqs**: Topics 0-1

---

## Why Linked Lists?

| | Array (Python list) | Linked List |
|---|---|---|
| Memory | Contiguous block | Scattered nodes |
| Access by index | O(1) — fast | O(n) — slow |
| Insert at front | O(n) — slow (shift all) | O(1) — fast |
| Insert at end | O(1) — fast | O(1) with tail pointer |
| Delete from middle | O(n) — slow (shift) | O(1) if you have the node |

**Analogy**: A linked list is like a scavenger hunt — each clue (node) tells you where the next clue is.

```
Array:        [10][20][30][40]     ← items sit side by side

Linked List:  [10]→[20]→[30]→[40]→None   ← each node points to the next
```

---
## Step 1: Building a Node and a Linked List from Scratch

In [ ]:
class ListNode:
    """A single node in a linked list."""
    def __init__(self, val=0, next=None):
        self.val = val    # the data
        self.next = next  # pointer to next node (or None)
    
    def __repr__(self):
        return f"ListNode({self.val})"


# Helper functions to make our life easier
def build_list(values):
    """Create a linked list from a Python list."""
    if not values:
        return None
    head = ListNode(values[0])
    current = head
    for val in values[1:]:
        current.next = ListNode(val)
        current = current.next
    return head

def print_list(head):
    """Print a linked list nicely."""
    parts = []
    current = head
    while current:
        parts.append(str(current.val))
        current = current.next
    print(" → ".join(parts) + " → None")

def to_list(head):
    """Convert linked list back to Python list."""
    result = []
    while head:
        result.append(head.val)
        head = head.next
    return result


# Let's build one!
head = build_list([10, 20, 30, 40, 50])
print_list(head)

# Access individual nodes
print(f"\nFirst node:  {head.val}")
print(f"Second node: {head.next.val}")
print(f"Third node:  {head.next.next.val}")

---
## Step 2: Basic Operations

### Traversal — Visiting Every Node

In [ ]:
# Traversal: visit each node one by one
def traverse(head):
    current = head
    while current:
        print(f"  Visiting node with value: {current.val}")
        current = current.next  # move to next node
    print("  Reached the end (None)")

head = build_list([10, 20, 30])
traverse(head)

# Count the length
def length(head):
    count = 0
    while head:
        count += 1
        head = head.next
    return count

print(f"\nLength: {length(build_list([10, 20, 30, 40]))}")

---
## Core Problem 1: Reverse a Linked List

**THE most important linked list operation.** Asked in almost every interview.

```
Before: 1 → 2 → 3 → 4 → 5 → None
After:  5 → 4 → 3 → 2 → 1 → None
```

**Idea**: Use 3 pointers — `prev`, `current`, `next_node`.  
At each step, flip the arrow direction.

In [ ]:
# Iterative approach
def reverse_list(head):
    prev = None
    current = head
    
    while current:
        next_node = current.next   # save the next node
        current.next = prev        # flip the arrow!
        prev = current             # advance prev
        current = next_node        # advance current
    
    return prev  # prev is the new head

# Test
head = build_list([1, 2, 3, 4, 5])
print("Before:", end=" "); print_list(head)
head = reverse_list(head)
print("After: ", end=" "); print_list(head)

In [ ]:
# Let's trace it step by step to really understand
head = build_list([1, 2, 3, 4])

prev = None
current = head
step = 0

print("Reversing [1 → 2 → 3 → 4 → None]:\n")
while current:
    step += 1
    next_node = current.next
    current.next = prev
    
    prev_str = str(prev.val) if prev else "None"
    next_str = str(next_node.val) if next_node else "None"
    print(f"Step {step}: current={current.val}, flip arrow to point at {prev_str}")
    print(f"         prev={current.val}, next move to {next_str}")
    
    prev = current
    current = next_node

print(f"\nDone! New head = {prev.val}")
print("Result: ", end=""); print_list(prev)

In [ ]:
# Recursive approach (for understanding recursion — we'll cover this in depth in Topic 6)
def reverse_list_recursive(head):
    if not head or not head.next:
        return head
    
    new_head = reverse_list_recursive(head.next)
    head.next.next = head  # flip the arrow
    head.next = None       # old head now points to None
    
    return new_head

head = build_list([1, 2, 3, 4, 5])
head = reverse_list_recursive(head)
print("Recursive reverse:", end=" "); print_list(head)

---
## Core Problem 2: Detect a Cycle (Floyd's Algorithm)

**Problem**: Determine if a linked list has a cycle (loop).  
**Idea**: Use two pointers — `slow` moves 1 step, `fast` moves 2 steps.  
If there's a cycle, they'll eventually meet. If not, `fast` hits None.

```
With cycle:      1 → 2 → 3 → 4 → 5
                              ↑       ↓
                              8 ← 7 ← 6

Without cycle:   1 → 2 → 3 → 4 → None
```

In [ ]:
def has_cycle(head):
    slow = head
    fast = head
    
    while fast and fast.next:
        slow = slow.next          # 1 step
        fast = fast.next.next     # 2 steps
        
        if slow == fast:
            return True
    
    return False

# Test 1: No cycle
head = build_list([1, 2, 3, 4, 5])
print(f"No cycle list: {has_cycle(head)}")  # False

# Test 2: Create a cycle manually
head = build_list([1, 2, 3, 4, 5])
# Make node 5 point back to node 3
node3 = head.next.next
node5 = node3.next.next
node5.next = node3  # cycle!
print(f"Cycle list: {has_cycle(head)}")     # True

---
## Core Problem 3: Merge Two Sorted Lists

```
List 1:  1 → 2 → 4
List 2:  1 → 3 → 4
Merged:  1 → 1 → 2 → 3 → 4 → 4
```

In [ ]:
def merge_two_lists(l1, l2):
    dummy = ListNode(0)  # dummy head to simplify edge cases
    tail = dummy
    
    while l1 and l2:
        if l1.val <= l2.val:
            tail.next = l1
            l1 = l1.next
        else:
            tail.next = l2
            l2 = l2.next
        tail = tail.next
    
    # Attach the remaining nodes
    tail.next = l1 if l1 else l2
    
    return dummy.next  # skip the dummy

l1 = build_list([1, 2, 4])
l2 = build_list([1, 3, 4])
print("List 1:", end=" "); print_list(l1)
print("List 2:", end=" "); print_list(l2)
merged = merge_two_lists(l1, l2)
print("Merged:", end=" "); print_list(merged)

### The Dummy Node Trick
Notice we used `dummy = ListNode(0)` — this is a very common trick in linked list problems.  
It avoids special-casing the first node. At the end, `dummy.next` is the real head.

---
## Core Problem 4: Remove Nth Node from End

In [ ]:
# Remove the Nth node from the end in ONE pass
# Trick: Use two pointers, n apart

def remove_nth_from_end(head, n):
    dummy = ListNode(0, head)
    fast = dummy
    slow = dummy
    
    # Move fast n+1 steps ahead
    for _ in range(n + 1):
        fast = fast.next
    
    # Move both until fast reaches end
    while fast:
        slow = slow.next
        fast = fast.next
    
    # slow is now right before the node to remove
    slow.next = slow.next.next
    
    return dummy.next

head = build_list([1, 2, 3, 4, 5])
print("Before:", end=" "); print_list(head)
head = remove_nth_from_end(head, 2)
print("Remove 2nd from end:", end=" "); print_list(head)  # 1→2→3→5

---
## Core Problem 5: Palindrome Linked List

In [ ]:
def is_palindrome_list(head):
    # Step 1: Find middle using slow/fast pointers
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    
    # Step 2: Reverse second half
    prev = None
    while slow:
        next_node = slow.next
        slow.next = prev
        prev = slow
        slow = next_node
    
    # Step 3: Compare first half with reversed second half
    left, right = head, prev
    while right:
        if left.val != right.val:
            return False
        left = left.next
        right = right.next
    
    return True

print(is_palindrome_list(build_list([1, 2, 2, 1])))    # True
print(is_palindrome_list(build_list([1, 2, 3, 2, 1]))) # True
print(is_palindrome_list(build_list([1, 2, 3])))        # False

---
## Core Problem 6: Add Two Numbers

In [ ]:
# Numbers are stored in reverse order
# 342 = 2 → 4 → 3
# 465 = 5 → 6 → 4
# 807 = 7 → 0 → 8

def add_two_numbers(l1, l2):
    dummy = ListNode(0)
    current = dummy
    carry = 0
    
    while l1 or l2 or carry:
        val1 = l1.val if l1 else 0
        val2 = l2.val if l2 else 0
        
        total = val1 + val2 + carry
        carry = total // 10
        digit = total % 10
        
        current.next = ListNode(digit)
        current = current.next
        
        l1 = l1.next if l1 else None
        l2 = l2.next if l2 else None
    
    return dummy.next

l1 = build_list([2, 4, 3])  # 342
l2 = build_list([5, 6, 4])  # 465
result = add_two_numbers(l1, l2)
print("342 + 465 =", end=" "); print_list(result)  # 7→0→8 = 807

---
## Practice Problems

| # | Problem | Difficulty | LeetCode |
|---|---------|------------|----------|
| 1 | Reverse Linked List | Easy | #206 |
| 2 | Merge Two Sorted Lists | Easy | #21 |
| 3 | Linked List Cycle | Easy | #141 |
| 4 | Remove Nth Node from End | Medium | #19 |
| 5 | Palindrome Linked List | Easy | #234 |
| 6 | Add Two Numbers | Medium | #2 |
| 7 | Intersection of Two Lists | Easy | #160 |
| 8 | Reorder List | Medium | #143 |

---

## Key Patterns Summary

```
LINKED LIST PATTERN CHEAT SHEET:

"Reverse"              → prev/current/next three-pointer dance
"Find middle"          → slow (1 step) & fast (2 steps)
"Detect cycle"         → Floyd's: slow & fast pointers
"Merge sorted lists"   → Dummy head + compare & attach
"Remove Nth from end"  → Two pointers, n apart
"Palindrome"           → Find middle + reverse second half + compare
```

### Next up: **Topic 4 — Stacks & Queues**